In [1]:
import os
os.environ['USER_AGENT'] = 'BA_GPT/1.0 (contact: walrus_z@hotmail.com)'

In [2]:
import os
import sys
import yaml
import types
import pprint
import logging
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat import EVENT_LOGGER_NAME, TRACE_LOGGER_NAME
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.tools import AgentTool
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.memory.canvas import TextCanvasMemory
from autogen_ext.models.openai import OpenAIChatCompletionClient
from opentelemetry import trace
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.instrumentation.openai import OpenAIInstrumentor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor


# Project import paths
# Get the current directory (which is project/test)
notebook_dir = os.getcwd()

# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

from agents_creator import create_agents_from_config
from rag_helpers import rag_indexer_runner
from model_loader import load_model_client


In [3]:
logging.basicConfig(level=logging.WARNING)

# For trace logging.
trace_logger = logging.getLogger(TRACE_LOGGER_NAME)
trace_logger.addHandler(logging.StreamHandler())
trace_logger.setLevel(logging.DEBUG)

# For structured message logging, such as low-level messages between agents.
event_logger = logging.getLogger(EVENT_LOGGER_NAME)
event_logger.addHandler(logging.StreamHandler())
event_logger.setLevel(logging.DEBUG)

In [ ]:
# Set up telemetry span exporter.
otel_exporter = OTLPSpanExporter(endpoint="http://localhost:4317", insecure=True)
span_processor = BatchSpanProcessor(otel_exporter)

# Set up telemetry trace provider.
tracer_provider = TracerProvider(resource=Resource({"service.name": "autogen-test-agentchat"}))
tracer_provider.add_span_processor(span_processor)
trace.set_tracer_provider(tracer_provider)

# Instrument the OpenAI Python library
OpenAIInstrumentor().instrument()

# we will get reference this tracer later using its service name
# tracer = trace.get_tracer("autogen-test-agentchat")

ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export trace

In [5]:
rag_vector_memory = await rag_indexer_runner('/run/media/jun/Jun 6T/Code/TFM_BA_GPT/lib/rag_config.yaml')

✅ All documents and URLs are already indexed.


In [6]:
# 1. Create the canvas memory and tool:
text_canvas_memory = TextCanvasMemory()
update_file_tool = text_canvas_memory.get_update_file_tool()

# 2. Register the tool as 'canvas.update_file_tool' for YAML import:
canvas = types.SimpleNamespace()
canvas.update_file_tool = update_file_tool
sys.modules['canvas'] = canvas  # This enables the tool import by your agent factory!

In [7]:
model_client = load_model_client("lmstudio")

In [8]:
print(vars(model_client))



{'_raw_config': {'model': 'google/gemma-3-27b', 'api_key': 'LMStudio_key', 'model_info': {'vision': False, 'function_calling': True, 'json_output': True, 'family': 'gemma', 'structured_output': True, 'multiple_system_messages': True}, 'base_url': 'http://127.0.0.1:1234/v1'}, '_client': <openai.AsyncOpenAI object at 0x7fe93877e270>, '_add_name_prefixes': False, '_include_name_in_message': True, '_model_info': {'vision': False, 'function_calling': True, 'json_output': True, 'family': 'gemma', 'structured_output': True, 'multiple_system_messages': True}, '_resolved_model': 'google/gemma-3-27b', '_create_args': {'model': 'google/gemma-3-27b'}, '_total_usage': RequestUsage(prompt_tokens=0, completion_tokens=0), '_actual_usage': RequestUsage(prompt_tokens=0, completion_tokens=0)}


In [9]:
# Helpful sanity prints:
print("raw:", getattr(model_client, "_raw_config", None))
print("create_args:", getattr(model_client, "_create_args", None))

raw: {'model': 'google/gemma-3-27b', 'api_key': 'LMStudio_key', 'model_info': {'vision': False, 'function_calling': True, 'json_output': True, 'family': 'gemma', 'structured_output': True, 'multiple_system_messages': True}, 'base_url': 'http://127.0.0.1:1234/v1'}
create_args: {'model': 'google/gemma-3-27b'}


In [10]:
# Construct the absolute path to project/lib/agents_config.yaml with user_tool
AGENTS_CONFIG = os.path.abspath(os.path.join(notebook_dir, '..', 'lib', 'agents_config_user_tool.yaml'))
with open(AGENTS_CONFIG, "r") as f:
        config_yaml = yaml.safe_load(f)
agent_names = list(config_yaml.keys())

user, agents = await create_agents_from_config(
    AGENTS_CONFIG,
    agent_names=agent_names,
    model_client=model_client,
    vector_memory=rag_vector_memory,
    canvas_memory=text_canvas_memory,
    input_func=input
)


In [11]:
# 1️⃣ Grab the existing writer agent
old_writer = agents["functional_spec_writer_agent"]

# 2️⃣ Wrap the sub-agents as tools
use_case_tool = AgentTool(agent=agents["use_case_agent"])
glossary_tool = AgentTool(agent=agents["glossary_agent"])

# 3️⃣ Collect the writer's current tools (if any)
existing_tools = getattr(old_writer, "_tools", []) or []

# 4️⃣ Combine tools: existing ones + new agent-tools
combined_tools = existing_tools + [use_case_tool, glossary_tool]

# 5️⃣ Extract the system message text (AutoGen stores it as a list of SystemMessage objects)
system_message_text = ""
if hasattr(old_writer, "_system_messages") and old_writer._system_messages:
    system_message_text = old_writer._system_messages[0].content  # first SystemMessage content

# 6️⃣ Rebuild functional_spec_writer_agent with combined tools
agents["functional_spec_writer_agent"] = AssistantAgent(
    name=old_writer._name,
    description=old_writer._description,
    model_client=old_writer._model_client,
    memory=old_writer._memory,
    tools=combined_tools,
    reflect_on_tool_use=old_writer._reflect_on_tool_use,
    system_message=system_message_text,
    model_client_stream=old_writer._model_client_stream,
)

print("✅ functional_spec_writer_agent rebuilt with use_case_tool and glossary_tool attached.")


✅ functional_spec_writer_agent rebuilt with use_case_tool and glossary_tool attached.


In [12]:
# 1) Grab the existing interviewer
old_interviewer = agents["interviewer_agent"]

# 2) Create the user_tool as an AgentTool pointing to the human user agent
#    (Name it explicitly to make the tool-call obvious in traces: `user_tool`)
user_tool = AgentTool(agent=user)

# 3) Collect the interviewer's current tools (if any)
existing_tools = getattr(old_interviewer, "_tools", []) or []

# 4) Combine tools: existing ones + user_tool
combined_tools = existing_tools + [user_tool]

# 5) Extract the interviewer's system message text
interviewer_system_msg = ""
if hasattr(old_interviewer, "_system_messages") and old_interviewer._system_messages:
    interviewer_system_msg = old_interviewer._system_messages[0].content

# 6) Ensure the interviewer’s system message instructs tool usage (replace if needed)
#    You can also update the YAML to already include the lines below, but this keeps it robust at runtime:
INTERVIEWER_TOOL_LINE = (
    "Use the user_tool to send your question to the human and wait for the reply. "
    "Only you may contact the human, and only via user_tool."
)
if "user_tool" not in interviewer_system_msg:
    interviewer_system_msg = (
        interviewer_system_msg.strip() + "\n\n" +
        "**CRITICAL ACCESS RULE**\n"
        "- Use the user_tool to ask the human exactly ONE question at a time and wait for the reply.\n"
        "- Only you may contact the human, and only via user_tool.\n"
    )

# 7) Rebuild interviewer with the new tool attached (pattern you used for the writer)
agents["interviewer_agent"] = AssistantAgent(
    name=old_interviewer._name,
    description=old_interviewer._description,
    model_client=old_interviewer._model_client,
    memory=old_interviewer._memory,
    tools=combined_tools,
    reflect_on_tool_use=getattr(old_interviewer, "_reflect_on_tool_use", False),
    system_message=interviewer_system_msg,
    model_client_stream=getattr(old_interviewer, "_model_client_stream", None),
)

print("✅ interviewer_agent rebuilt with user_tool attached and system message updated.")

✅ interviewer_agent rebuilt with user_tool attached and system message updated.


In [13]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    pprint.pprint(vars(agent))




--- Agent: planning_agent ---
{'_description': 'The orchestrator agent responsible for breaking down a new '
                 'task into subtasks, assigning them to the right agents, and '
                 'ensuring the workflow progresses smoothly until completion.',
 '_handoff_tools': [],
 '_handoffs': {},
 '_is_running': False,
 '_max_tool_iterations': 1,
 '_memory': None,
 '_metadata': {},
 '_model_client': <autogen_ext.models.openai._openai_client.OpenAIChatCompletionClient object at 0x7fe93877e120>,
 '_model_client_stream': True,
 '_model_context': <autogen_core.model_context._unbounded_chat_completion_context.UnboundedChatCompletionContext object at 0x7fe93877ecf0>,
 '_name': 'planning_agent',
 '_output_content_type': None,
 '_output_content_type_format': None,
 '_reflect_on_tool_use': False,
 '_structured_message_factory': None,
 '_system_messages': [SystemMessage(content='You are the PLANNING AGENT, the coordinator for this multi-agent system.\n\n**YOUR MISSION**\n  Break dow

In [14]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    print("Tools:")
    for tool in agent._tools:
        print(f"Tool Name: {tool.name}")
        print(f"Description: {tool.description}\n")



--- Agent: planning_agent ---
Tools:

--- Agent: interviewer_agent ---
Tools:
Tool Name: save_conversation_to_jsonl
Description: 
Saves a single conversation turn (speaker + message) to a jsonl file.

Args:
    speaker (str): Who is speaking (e.g., "interviewer" or "user").
    message (str): The content of the message.


Tool Name: user
Description: A user to asnwer questions in the workflow.


--- Agent: summarizer_agent ---
Tools:

--- Agent: functional_spec_writer_agent ---
Tools:
Tool Name: write_md_file_tool
Description: 
Writes the functional specification to a Markdown file.


Tool Name: use_case_agent
Description: An agent dedicated to generating Use Cases and User Stories from the project information.

Tool Name: glossary_agent
Description: An agent dedicated to building a Glossary of terms from the gathered requirements and project information.


--- Agent: functional_spec_checker_agent ---
Tools:
Tool Name: read_md_file_tool
Description: 
Reads the entire content of a Mark

In [15]:
selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
"""

In [16]:
# Order agents (workflow order), no user_proxy last as this is for user_tool workflow
agent_order = [
    "planning_agent",
    "interviewer_agent",
    "summarizer_agent",
    "functional_spec_writer_agent",
    "functional_spec_checker_agent",
    "diagram_creator_agent",
    "mermaid_code_reviewer_agent",
]
member_agents = [agents[name] for name in agent_order if name in agents]

In [17]:
termination = MaxMessageTermination(100000000) | TextMentionTermination("TERMINATE")

# --- Use SelectorGroupChat ---
team = SelectorGroupChat(
    member_agents,
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=False
)

In [18]:
task = "Interview a customer to collect requirements and scope of a new product then write the functional specifications"

In [19]:
runtime = SingleThreadedAgentRuntime(
            tracer_provider=trace.NoOpTracerProvider(),  # Disable telemetry for runtime.
        )
runtime.start()

In [ ]:
await Console(team.run_stream(task=task))

await runtime.stop()

await model_client.close()

Model selected a valid name: planning_agent (attempt 1)
DEBUG:autogen_agentchat:Model selected a valid name: planning_agent (attempt 1)
Selected speaker: planning_agent
DEBUG:autogen_agentchat:Selected speaker: planning_agent


---------- TextMessage (user) ----------
Interview a customer to collect requirements and scope of a new product then write the functional specifications
---------- ModelClientStreamingChunkEvent (planning_agent) ----------
Okay, let's begin! Here's the initial breakdown of tasks and assignment:

**Task List:**
interviewer_agent: pending
summarizer_agent: not started
functional_spec_writer_agent: not started
functional_spec_checker_agent: not started
diagram_creator_agent: not started
mermaid_code_reviewer_agent: not started

1. interviewer_agent : Start gathering requirements from the user.



Model selected a valid name: interviewer_agent (attempt 1)
DEBUG:autogen_agentchat:Model selected a valid name: interviewer_agent (attempt 1)
Selected speaker: interviewer_agent
DEBUG:autogen_agentchat:Selected speaker: interviewer_agent


---------- MemoryQueryEvent (interviewer_agent) ----------
[MemoryContent(content='I have been an analyst for a little while now and I have to say this is a perfectly succinct starting point for requirements gathering! Well done\nReply \n\n\n\n\n\n\n \n\nToni \n\t\t\t\t\ton June 13, 2014 at 1:52 PM\t\t\t\t\n\n\n\nIt’s always interesting to see how others approach requirements gathering. Thanks for the information.\nReply \n\n\n\n\n\n\n \n\nAngelo the BA \n\t\t\t\t\ton September 9, 2014 at 10:55 PM', mime_type='MemoryMimeType.TEXT', metadata={'source': 'https://practicalanalyst.com/requirements-elicitation-most-valuable-questions/', 'mime_type': 'MemoryMimeType.TEXT', 'chunk_index': 25, 'score': 0.5219577550888062, 'id': '2eb4ba3c-0d5b-496a-9010-955dcfcfee91'}), MemoryContent(content='You’ll discover bits of what their answers should include.17. What Is Your Requirement Gathering Process?Every business analyst has a unique requirements-gathering process. Our demo videos show how our pro

id='d591734e-486d-44d2-9935-dfb6bdf7214c' source='interviewer_agent' models_usage=RequestUsage(prompt_tokens=0, completion_tokens=0) metadata={} created_at=datetime.datetime(2025, 8, 8, 17, 18, 51, 752937, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='289480503', arguments='{"task": "Can you briefly describe the product you envision? What problem does it solve, and for whom?"}', name='user')] type='ToolCallRequestEvent'
id='d591734e-486d-44d2-9935-dfb6bdf7214c' source='interviewer_agent' models_usage=RequestUsage(prompt_tokens=0, completion_tokens=0) metadata={} created_at=datetime.datetime(2025, 8, 8, 17, 18, 51, 752937, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='289480503', arguments='{"task": "Can you briefly describe the product you envision? What problem does it solve, and for whom?"}', name='user')] type='ToolCallRequestEvent'
DEBUG:autogen_agentchat.events:id='d591734e-486d-44d2-9935-dfb6bdf7214c' source='interviewer_agent' models_usage=RequestUsage(prompt


---------- ToolCallRequestEvent (interviewer_agent) ----------
[FunctionCall(id='289480503', arguments='{"task": "Can you briefly describe the product you envision? What problem does it solve, and for whom?"}', name='user')]
---------- TextMessage (user) ----------
Can you briefly describe the product you envision? What problem does it solve, and for whom?


... (54 outputs left out)

id='ed05afa6-f16a-47a3-adbf-4d796693efc1' source='interviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 8, 8, 17, 20, 54, 297920, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='Saved message from interviewer to ../conversation/interview.jsonl', name='save_conversation_to_jsonl', call_id='789313309', is_error=False)] type='ToolCallExecutionEvent'
DEBUG:autogen_agentchat.events:id='ed05afa6-f16a-47a3-adbf-4d796693efc1' source='interviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 8, 8, 17, 20, 54, 297920, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='Saved message from interviewer to ../conversation/interview.jsonl', name='save_conversation_to_jsonl', call_id='789313309', is_error=False)] type='ToolCallExecutionEvent'


---------- ToolCallExecutionEvent (interviewer_agent) ----------
[FunctionExecutionResult(content='Saved message from interviewer to ../conversation/interview.jsonl', name='save_conversation_to_jsonl', call_id='789313309', is_error=False)]
---------- ModelClientStreamingChunkEvent (interviewer_agent) ----------
Product Overview: pending
Stakeholders & Users: pending
KPI & Success Metrics: pending
Scope In and Out: pending
Functional Requirements: pending
Non-Functional Requirements: pending
User Roles & Personas: pending
Integrations & External Systems: pending
Constraints & Assumptions: not started
Integrations & Data: not started
Dependencies & Risks: not started
Open Questions: not started

[TOOL_REQUEST]
{"name": "user", "arguments": {"task":"Does this product need to integrate with any existing systems? If so, please describe those integrations and the data that needs to be exchanged."}}
[END_TOOL_REQUEST]


Model selected a valid name: planning_agent (attempt 1)
DEBUG:autogen_agentchat:Model selected a valid name: planning_agent (attempt 1)
Selected speaker: planning_agent
DEBUG:autogen_agentchat:Selected speaker: planning_agent


---------- ModelClientStreamingChunkEvent (planning_agent) ----------
Okay, I understand the interviewer agent is now gathering information about **Integrations & External Systems**. I remain in a waiting state, prepared to summarize once enough information has been gathered. 

I will prioritize building out the **Integrations & External Systems** section when that data becomes available. I'm ready to synthesize the collected requirements into a structured summary as soon as possible.

Product Overview: pending
Stakeholders & Users: pending
KPI & Success Metrics: pending
Scope In and Out: pending
Functional Requirements: pending
Non-Functional Requirements: pending
User Roles & Personas: pending
Integrations & External Systems: pending
Constraints & Assumptions: pending
Integrations & Data: not started
Dependencies & Risks: not started
Open Questions: not started

[TOOL_REQUEST]
{"name": "user", "arguments": {"task":"What constraints or assumptions are we operating under? (e.g.,